In [6]:
import pandas as pd
from data import *
from data_utils import build_fuzzy_correspondence

Correspondances trouvées : 550


/home/onyxia/work/OT_simulation_maladies/Embeddings/data.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  profils_omim = profils_omim.reset_index()


gene_omim :  (2247, 11614)
gene_orpha :  (1700, 11614)
df1_omim : (2247, 8)
 df1_orpha : (1700, 8)


In [4]:
# TEST table d'équivalences
import requests, xml.etree.ElementTree as ET
import pandas as pd

url = "https://www.orphadata.com/data/xml/en_product1.xml"
response = requests.get(url)
response.raise_for_status()
tree = ET.fromstring(response.content)
rows = []
for disorder in tree.iter("Disorder"):
    orpha_id = "ORPHA:" + disorder.findtext("OrphaCode")
    for ref in disorder.iter("ExternalReference"):
        if ref.findtext("Source") == "OMIM":
            rows.append({
                "orpha_id": orpha_id,
                "omim_id":  "OMIM:" + ref.findtext("Reference"),
                "mapping_type": ref.findtext("DisorderMappingRelation/Name")
            })

df_orpha_omim = pd.DataFrame(rows)

In [7]:
list_omim = df_orpha_omim['omim_id'].unique()
list_orpha = df_orpha_omim['orpha_id'].unique()
len(list_omim), len(list_orpha)

work_omim = df_pivot[df_pivot['database_id'].isin(list_omim)]
work_orpha = df_pivot[df_pivot['database_id'].isin(list_orpha)]

work_omim.shape, work_orpha.shape

((6562, 11614), (3189, 11614))

In [20]:
df1_orpha_bis = pd.merge(work_orpha, df1, how='left', left_on='database_id', right_on='disease_id')
# print(df1_orpha.isna().sum())
print(df1_orpha_bis.shape)
df1_omim_bis = pd.merge(work_omim, df1, how='left', left_on='database_id', right_on='disease_id')
print(df1_omim_bis.shape) #, df1_omim.isna().sum())

(230606, 11622)
(261836, 11622)


In [21]:
df1_omim_bis = df1_omim_bis.groupby("disease_id", as_index=False, dropna=True).agg(
    ncbi_gene_id=("ncbi_gene_id", "first"),
    gene_symbol=("gene_symbol", "first"),
    association_type=("association_type", "first"),
    protein=("#string_protein_id", "first"),
    annotation=("annotation", "first"),
    protein2=("protein2", list),
    combined_score=("combined_score", list)
)

In [22]:
df1_omim_bis.shape

(5474, 8)

In [23]:
df1_orpha_bis = df1_orpha_bis.groupby("disease_id", as_index=False, dropna=True).agg(
    ncbi_gene_id=("ncbi_gene_id", "first"),
    gene_symbol=("gene_symbol", "first"),
    association_type=("association_type", "first"),
    protein=("#string_protein_id", "first"),
    annotation=("annotation", "first"),
    protein2=("protein2", list),
    combined_score=("combined_score", list)
)
df1_orpha_bis.shape

(2275, 8)

In [25]:
df1_orpha_bis

,disease_id,ncbi_gene_id,gene_symbol,association_type,protein,annotation,protein2,combined_score
0,ORPHA:100,NCBIGene:472,ATM,UNKNOWN,9606.ENSP00000278616,Serine-protein kinase ATM; Serine/threonine pr...,"[9606.ENSP00000360025, 9606.ENSP00000384849, 9...","[746.0, 807.0, 991.0, 897.0, 921.0, 858.0, 777..."
1,ORPHA:1000,NCBIGene:8943,AP3D1,UNKNOWN,9606.ENSP00000495274,AP-3 complex subunit delta-1; Part of the AP-3...,"[9606.ENSP00000325369, 9606.ENSP00000470176, 9...","[999.0, 860.0, 799.0, 821.0, 896.0, 844.0, 999..."
2,ORPHA:100006,NCBIGene:351,APP,UNKNOWN,9606.ENSP00000284981,Gamma-secretase C-terminal fragment 50; Functi...,"[9606.ENSP00000296861, 9606.ENSP00000498587, 9...","[998.0, 886.0, 950.0, 773.0, 837.0, 969.0, 943..."
3,ORPHA:100008,NCBIGene:1471,CST3,UNKNOWN,9606.ENSP00000381448,Cystatin-C; As an inhibitor of cysteine protei...,"[9606.ENSP00000360687, 9606.ENSP00000380432, 9...","[749.0, 761.0, 736.0, 951.0, 924.0, 739.0, 727..."
4,ORPHA:100050,NCBIGene:710,SERPING1,UNKNOWN,9606.ENSP00000278407,Plasma protease C1 inhibitor; Activation of th...,"[9606.ENSP00000363773, 9606.ENSP00000363079, 9...","[973.0, 979.0, 964.0, 759.0, 980.0, 774.0, 755..."
...,...,...,...,...,...,...,...,...
2270,ORPHA:99956,NCBIGene:81846,SBF2,UNKNOWN,9606.ENSP00000256190,Myotubularin-related protein 13; Guanine nucle...,"[9606.ENSP00000230124, 9606.ENSP00000261263, 9...","[741.0, 816.0, 765.0, 741.0, 719.0, 714.0, 786..."
2271,ORPHA:99966,NCBIGene:6598,SMARCB1,UNKNOWN,9606.ENSP00000340883,SWI/SNF-related matrix-associated actin-depend...,"[9606.ENSP00000462355, 9606.ENSP00000341805, 9...","[853.0, 999.0, 901.0, 859.0, 853.0, 843.0, 853..."
2272,ORPHA:99967,NCBIGene:2521,FUS,UNKNOWN,9606.ENSP00000254108,RNA-binding protein FUS; DNA/RNA-binding prote...,"[9606.ENSP00000483254, 9606.ENSP00000221419, 9...","[918.0, 927.0, 985.0, 872.0, 982.0, 757.0, 888..."
2273,ORPHA:99976,NCBIGene:2064,ERBB2,UNKNOWN,9606.ENSP00000269571,Receptor tyrosine-protein kinase erbB-2; Prote...,"[9606.ENSP00000352414, 9606.ENSP00000222254, 9...","[815.0, 985.0, 729.0, 808.0, 994.0, 904.0, 811..."


In [16]:
len(set(df1['disease_id'].unique())&set(work_omim['database_id']))
len(set(df1['disease_id'].unique())&set(work_orpha['database_id']))

2275

In [8]:
res = set(genes_to_disease['gene_symbol'])&set(doc['gene_symbol'])
print(len(res))
print(len(genes_to_disease['gene_symbol'].unique()))
print(len(set(ppi['protein1'])&set(doc[doc['gene_symbol'].isin(res)]['#string_protein_id'])))


5361
5505
5193


In [5]:
print(f"Protéines dans df0 et pas dans ppi : {len(set(df0["#string_protein_id"]))-len(set(df0["#string_protein_id"])&set(ppi["protein1"]))}")

Protéines dans df0 et pas dans ppi : 169


In [5]:
print(len(set(df1['disease_id'])&set(df_hpoa['database_id'])))
print(len(df_hpoa['database_id'].unique()))
print(len(df1['disease_id'].unique()))
print(len(set(genes_to_disease['disease_id'].unique())&set(df_hpoa['database_id'].unique())))

4688
12996
4928
9120


In [9]:
print("========= Maladies en commun avec df1 ==========")
print("Dataset : profils_omim")
print(f"{len(set(profils_omim['disease'].unique())&set(df1['disease_id'].unique()))}/{len(profils_omim['disease'].unique())}")
print("Dataset : df_omim")
print(f"{len(set(df_omim['database_id'].unique())&set(df1['disease_id'].unique()))}/{len(df_omim['database_id'].unique())}")
print("Dataset : df_orpha")
print(f"{len(set(df_orpha['database_id'].unique())&set(df1['disease_id'].unique()))}/{len(df_orpha['database_id'].unique())}")


========= Maladies en commun avec df1 ==========
Dataset : profils_omim
4158/6139
Dataset : df_omim
270/550
Dataset : df_orpha
14/550


In [10]:
print(len(set(df_omim['database_id'].unique())&set(genes_to_disease['disease_id'].unique())))
print(len(set(df_orpha['database_id'].unique())&set(genes_to_disease['disease_id'].unique())))

418
397


In [11]:
genes_correspondence={}
for i in range(correspondence_exacte.shape[0]):
    omim = correspondence_exacte["omim_id"][i]
    orpha = correspondence_exacte["orpha_id"][i]
    if omim in genes_to_disease["disease_id"].values:
        gene_omim=genes_to_disease[genes_to_disease['disease_id']==omim]['gene_symbol']
    else:
        gene_omim=None
    if orpha in genes_to_disease["disease_id"].values:
        gene_orpha=genes_to_disease[genes_to_disease['disease_id']==orpha]['gene_symbol']
    else:
        gene_orpha=None
    genes_correspondence[(omim, orpha)]=(gene_omim, gene_orpha)

print("Nombre de maladies correspondantes qui n'ont pas exactement les mêmes gènes :", 
sum(
    v[0] is not None and v[1] is not None and set(v[0]) != set(v[1])
    for v in genes_correspondence.values()
))
print([k for k,v in genes_correspondence.items() if v[0] is not None and v[1] is not None and set(v[0]) != set(v[1])])
print("Nombre de paires qui ont au moins un None :", len(genes_correspondence)-
sum(v[0] is not None and v[1] is not None for v in genes_correspondence.values()))

Nombre de maladies correspondantes qui n'ont pas exactement les mêmes gènes : 60
[('OMIM:268000', 'ORPHA:791'), ('OMIM:263800', 'ORPHA:358'), ('OMIM:601321', 'ORPHA:638'), ('OMIM:219000', 'ORPHA:2052'), ('OMIM:615237', 'ORPHA:2301'), ('OMIM:612376', 'ORPHA:520'), ('OMIM:242600', 'ORPHA:42062'), ('OMIM:167400', 'ORPHA:46348'), ('OMIM:269250', 'ORPHA:3144'), ('OMIM:218330', 'ORPHA:1515'), ('OMIM:601859', 'ORPHA:3261'), ('OMIM:273800', 'ORPHA:849'), ('OMIM:225500', 'ORPHA:289'), ('OMIM:254500', 'ORPHA:29073'), ('OMIM:143100', 'ORPHA:399'), ('OMIM:277590', 'ORPHA:3447'), ('OMIM:603554', 'ORPHA:39041'), ('OMIM:236730', 'ORPHA:2704'), ('OMIM:123150', 'ORPHA:1540'), ('OMIM:117550', 'ORPHA:821'), ('OMIM:229200', 'ORPHA:90354'), ('OMIM:606764', 'ORPHA:44890'), ('OMIM:208150', 'ORPHA:994'), ('OMIM:214800', 'ORPHA:138'), ('OMIM:254940', 'ORPHA:1358'), ('OMIM:600880', 'ORPHA:131'), ('OMIM:608572', 'ORPHA:1200'), ('OMIM:102370', 'ORPHA:969'), ('OMIM:219700', 'ORPHA:586'), ('OMIM:216340', 'ORPHA:347

In [3]:
df_hpoa

,database_id,disease_name,qualifier,hpo_id,reference,evidence,onset,frequency,sex,modifier,aspect,biocuration
0,OMIM:619340,developmental and epileptic encephalopathy 96,NaN,HP:0011097,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
1,OMIM:619340,developmental and epileptic encephalopathy 96,NaN,HP:0002187,PMID:31675180,PCS,NaN,1/1,NaN,NaN,P,HPO:probinson[2021-06-21]
2,OMIM:619340,developmental and epileptic encephalopathy 96,NaN,HP:0001518,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
3,OMIM:619340,developmental and epileptic encephalopathy 96,NaN,HP:0032792,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
4,OMIM:619340,developmental and epileptic encephalopathy 96,NaN,HP:0011451,PMID:31675180,PCS,NaN,1/2,NaN,NaN,P,HPO:probinson[2021-06-21]
...,...,...,...,...,...,...,...,...,...,...,...,...
282718,ORPHA:1777,temtamy syndrome,NaN,HP:0000324,ORPHA:1777,TAS,NaN,HP:0040283,NaN,NaN,P,ORPHA:orphadata[2026-02-16]
282719,ORPHA:1777,temtamy syndrome,NaN,HP:0000506,ORPHA:1777,TAS,NaN,HP:0040283,NaN,NaN,P,ORPHA:orphadata[2026-02-16]
282720,ORPHA:1777,temtamy syndrome,NaN,HP:0000568,ORPHA:1777,TAS,NaN,HP:0040283,NaN,NaN,P,ORPHA:orphadata[2026-02-16]
282721,ORPHA:1777,temtamy syndrome,NaN,HP:0004209,ORPHA:1777,TAS,NaN,HP:0040283,NaN,NaN,P,ORPHA:orphadata[2026-02-16]


In [5]:
correspondence95 = build_fuzzy_correspondence(df_hpoa, 0.9)

Correspondances exactes (=1.0)  : 553
Correspondances >= 0.9      : 1622


In [6]:
correspondence95

,omim_id,orpha_id,omim_name,orpha_name,similarity
24,OMIM:613490,ORPHA:60,alpha 1 antitrypsin deficiency,alpha 1 antitrypsin deficiency,1.0000
1621,OMIM:106750,ORPHA:69125,anonychia with flexural pigmentation,anonychia with flexural pigmentation,1.0000
0,OMIM:117650,ORPHA:1393,cerebrocostomandibular syndrome,cerebrocostomandibular syndrome,1.0000
21,OMIM:175200,ORPHA:2869,peutz jeghers syndrome,peutz jeghers syndrome,1.0000
2,OMIM:268000,ORPHA:791,retinitis pigmentosa,retinitis pigmentosa,1.0000
...,...,...,...,...,...
31,OMIM:617769,ORPHA:98755,spinocerebellar ataxia 45,spinocerebellar ataxia type 1,0.9004
1506,OMIM:618093,ORPHA:98755,spinocerebellar ataxia 48,spinocerebellar ataxia type 1,0.9003
459,OMIM:169500,ORPHA:99027,"leukodystrophy, adult onset, autosomal dominant",adult onset autosomal dominant leukodystrophy,0.9003
981,OMIM:616219,ORPHA:45358,"fibrosis of extraocular muscles, congenital, 5",congenital fibrosis of extraocular muscles,0.9002


In [ ]:
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print(f"Nombre de maladies ORPHA : {len(genes_to_disease[genes_to_disease['disease_id'].str.startswith("ORPHA")])}")
print(f"Nombre de maladies OMIM : {len(genes_to_disease[genes_to_disease['disease_id'].str.startswith("OMIM")])}")

def find_gene_correspondence(df, df_hpoa, disease_col, gene_col, threshold):
    genes2diseases = defaultdict(list)
    for i, row in df.iterrows():
        genes2diseases[row[gene_col]].append(row[disease_col])

    omim  = (df_hpoa[df_hpoa['database_id'].str.startswith('OMIM:')]
             [['disease_name', 'database_id']].drop_duplicates())
    orpha = (df_hpoa[df_hpoa['database_id'].str.startswith('ORPHA:')]
             [['disease_name', 'database_id']].drop_duplicates())

    omim_names  = omim['disease_name'].str.lower().str.strip()
    orpha_names = orpha['disease_name'].str.lower().str.strip()

    vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5))
    vectorizer.fit(pd.concat([omim_names, orpha_names]))
    omim_vecs  = vectorizer.transform(omim_names)
    orpha_vecs = vectorizer.transform(orpha_names)
    sim_matrix = cosine_similarity(omim_vecs, orpha_vecs)  # (n_omim, n_orpha)

    omim_id_to_idx = {row.database_id: i for i, row in enumerate(omim.itertuples())}
    orpha_id_to_idx = {row.database_id: j for j, row in enumerate(orpha.itertuples())}

    gene_sets = df.groupby(disease_col)[gene_col].apply(set)
    omim_sets = gene_sets[gene_sets.index.str.startswith("OMIM")]
    
    matches = []
    for omim_id, omim_genes in omim_sets.items():
        shared_diseases = set()
        if len(omim_genes)==1:
            (gene,) = omim_genes
            shared_diseases.update(genes2diseases[gene])  # Maladies partageant le même gène
        else:
            continue
        orpha_candidates = [d for d in shared_diseases if d.startswith("ORPHA")]  # Orpha partageant le même gène
        if not orpha_candidates:
            continue
        omim_idx = omim_id_to_idx.get(omim_id)
        if omim_idx is None:
            continue
        for orpha_id in orpha_candidates:
            orpha_idx = orpha_id_to_idx.get(orpha_id)
            if orpha_idx is None:
                continue
            score = sim_matrix[omim_idx, orpha_idx]
            if score >= threshold:
                matches.append({
                    "omim_id" : omim_id,
                    "orpha_id" : orpha_id,
                    "omim_name" : omim.iloc[omim_idx]['disease_name'],
                    "orpha_name" : orpha.iloc[orpha_idx]['disease_name'],
                    "genes" : ", ".join(sorted(omim_genes)),
                    "similarity" : round(float(score), 4),
                })
    df_matches = (pd.DataFrame(matches)
                  .drop_duplicates(subset=["omim_id", "orpha_id"])
                  .sort_values("similarity", ascending=False)
                  .reset_index(drop=True))

    print(f"Paires trouvées (gène + nom) : {len(df_matches)}")
    print(f"dont similarité = 1.0 : {(df_matches['similarity'] == 1.0).sum()}")
    return pd.DataFrame(matches)
    
df_gene_correspondence=find_gene_correspondence(genes_to_disease, df_hpoa, 'disease_id', 'gene_symbol', 0.85)


Nombre de maladies ORPHA : 8290
Nombre de maladies OMIM : 7624
Paires trouvées (gène + nom) : 1313
dont similarité = 1.0 : 351


In [11]:
print(df_gene_correspondence.shape)
print(len(set(correspondence_exacte['omim_id'].unique())&set(df_gene_correspondence['omim_id'].unique())))
print(len(set(df_gene_correspondence['orpha_id'].unique())&set(df_hpoa['database_id'].unique())))
df_gene_correspondence0=df_gene_correspondence.loc[df_gene_correspondence['orpha_id'].isin(df_hpoa['database_id'])]
print(df_gene_correspondence0.shape)

(6375, 3)
365
1700
(3858, 3)


In [29]:
list_omim = df_gene_correspondence['omim_id'].unique()
list_orpha = df_gene_correspondence['orpha_id'].unique()
len(list_omim), len(list_orpha)

work_omim = df_pivot[df_pivot['database_id'].isin(list_omim)]
work_orpha = df_pivot[df_pivot['database_id'].isin(list_orpha)]

work_omim.shape, work_orpha.shape

((1270, 11614), (807, 11614))

In [14]:
df1_orpha = pd.merge(work_orpha, df1, how='left', left_on='database_id', right_on='disease_id')
# print(df1_orpha.isna().sum())
print(df1_orpha.shape)
df1_omim = pd.merge(work_omim, df1, how='left', left_on='database_id', right_on='disease_id')
print(df1_omim.shape) #, df1_omim.isna().sum())

(79878, 11622)
(108575, 11622)


In [22]:
df1_omim

,database_id,HP:0000002,HP:0000003,HP:0000006,HP:0000007,HP:0000008,HP:0000009,HP:0000010,HP:0000011,HP:0000012,...,HP:6001440,HP:6001454,ncbi_gene_id,gene_symbol,association_type,disease_id,#string_protein_id,annotation,protein2,combined_score
0,OMIM:100100,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NCBIGene:1131,CHRM3,MENDELIAN,OMIM:100100,9606.ENSP00000255380,Muscarinic acetylcholine receptor M3; The musc...,9606.ENSP00000493985,903.0
1,OMIM:100100,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NCBIGene:1131,CHRM3,MENDELIAN,OMIM:100100,9606.ENSP00000255380,Muscarinic acetylcholine receptor M3; The musc...,9606.ENSP00000078429,908.0
2,OMIM:100100,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NCBIGene:1131,CHRM3,MENDELIAN,OMIM:100100,9606.ENSP00000255380,Muscarinic acetylcholine receptor M3; The musc...,9606.ENSP00000262958,717.0
3,OMIM:100100,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NCBIGene:1131,CHRM3,MENDELIAN,OMIM:100100,9606.ENSP00000255380,Muscarinic acetylcholine receptor M3; The musc...,9606.ENSP00000319713,715.0
4,OMIM:100100,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NCBIGene:1131,CHRM3,MENDELIAN,OMIM:100100,9606.ENSP00000255380,Muscarinic acetylcholine receptor M3; The musc...,9606.ENSP00000360021,910.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108570,OMIM:621485,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NCBIGene:55157,DARS2,MENDELIAN,OMIM:621485,9606.ENSP00000497569,"Aspartate--tRNA ligase, mitochondrial; asparty...",9606.ENSP00000307567,784.0
108571,OMIM:621485,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NCBIGene:55157,DARS2,MENDELIAN,OMIM:621485,9606.ENSP00000497569,"Aspartate--tRNA ligase, mitochondrial; asparty...",9606.ENSP00000360483,766.0
108572,OMIM:621485,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NCBIGene:55157,DARS2,MENDELIAN,OMIM:621485,9606.ENSP00000497569,"Aspartate--tRNA ligase, mitochondrial; asparty...",9606.ENSP00000355889,895.0
108573,OMIM:621485,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NCBIGene:55157,DARS2,MENDELIAN,OMIM:621485,9606.ENSP00000497569,"Aspartate--tRNA ligase, mitochondrial; asparty...",9606.ENSP00000199389,802.0


In [ ]:
df1_omim = df1_omim.groupby("disease_id", as_index=False, dropna=True).agg(
    ncbi_gene_id=("ncbi_gene_id", "first"),
    gene_symbol=("gene_symbol", "first"),
    association_type=("association_type", "first"),
    protein=("#string_protein_id", "first"),
    annotation=("annotation", "first"),
    protein2=("protein2", list),
    combined_score=("combined_score", list)
)

KeyError: "Label(s) ['#string_protein_id'] do not exist"

In [26]:
df1_omim.isna().sum()

disease_id           0
ncbi_gene_id         0
gene_symbol          0
association_type     0
protein             28
annotation          28
protein2             0
combined_score       0
dtype: int64

In [ ]:
df1_orpha = df1_orpha.groupby("disease_id", as_index=False, dropna=True).agg(
    ncbi_gene_id=("ncbi_gene_id", "first"),
    gene_symbol=("gene_symbol", "first"),
    association_type=("association_type", "first"),
    protein=("#string_protein_id", "first"),
    annotation=("annotation", "first"),
    protein2=("protein2", list),
    combined_score=("combined_score", list)
)
df1_orpha.shape

(1700, 8)

In [28]:
df1_orpha.isna().sum()

disease_id           0
ncbi_gene_id         0
gene_symbol          0
association_type     0
protein             21
annotation          21
protein2             0
combined_score       0
dtype: int64